In [95]:
import torch
import tiktoken
import torch.nn as nn
import torch.nn.functional as F
import math
if torch.mps.is_available():
    device = "mps"
else:
    raise Exception("no mps")

encoder = tiktoken.encoding_for_model("gpt-2")
print(encoder)


<Encoding 'gpt2'>


In [2]:
data = ""
with open("./data/verdict.txt") as fp:
    data = fp.read()

In [ ]:
CONTEXT_LENGTH = 64
VOCAB_SIZE = encoder.n_vocab
BATCH_SIZE = 32
EMBEDDING_DIMENSION = 128
NUM_HEADS = 4
NUM_EXPERTS = 4
encodings = encoder.encode(data)
print(len(encodings))
SIZE = (len(encodings)-CONTEXT_LENGTH-1, CONTEXT_LENGTH)

5145


In [ ]:

xs = torch.empty(size=SIZE, dtype=torch.int32)
ys = torch.empty(size=SIZE, dtype=torch.int32)

for i in range((len(encodings)-CONTEXT_LENGTH-1)):
    x = torch.tensor(encodings[i: i+CONTEXT_LENGTH])
    y = torch.tensor(encodings[i+1: i+1+CONTEXT_LENGTH])
    if len(x) != CONTEXT_LENGTH:
        raise Exception(f"len(x) should be {CONTEXT_LENGTH}")
    if len(y) != CONTEXT_LENGTH:
        raise Exception(f"len(y) should be {CONTEXT_LENGTH}")    
    xs[i] = x
    ys[i] = y


In [15]:
assert xs.shape == SIZE
assert ys.shape == SIZE

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(xs, ys)
dataloader = DataLoader(dataset=dataset, shuffle=False, batch_size=BATCH_SIZE)

torch.Size([32, 64])

In [ ]:
class CasualAttention(nn.Module):
    def __init__(self, head_dim:int, embedding_dimension:int) -> None:
        super().__init__()
        self.head_dim = head_dim
        self.query = nn.Linear(embedding_dimension, head_dim, bias=False)
        self.key = nn.Linear(embedding_dimension, head_dim, bias=False)
        self.value = nn.Linear(embedding_dimension, head_dim, bias=False)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # shape of x here is (B, CONTENT_LENGTH, EMBEDDING_DIM)
        q, k, v = self.query(x), self.key(x), self.value(x) 
        attn_scores = (q @ k.tanspose(-2, -1)) + mask # shape (B, CONTENT_LENGTH, CONTENT_LENGTH)
        return F.softmax(attn_scores/math.sqrt(self.head_dim), dim=-1) @ v

        


class MultiHeadCasualAttention(nn.Module):
    def __init__(self, num_heads: int, embedding_dimension:int, context_length:int) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.embedding_dimension = embedding_dimension
        self.context_length = context_length

        assert self.embedding_dimension % self.num_heads == 0, "embedding dimension should be divisible by num heads"
        self.head_dim =  self.embedding_dimension // self.num_heads

        _mask = torch.triu(torch.ones(size=(self.context_length, self.context_length)), diagonal=1)
        self.register_buffer("mask", torch.where(_mask == 1, -math.inf, 0), persistent=False)
        self.maksed_attns = [
            CasualAttention(head_dim=self.head_dim, embedding_dimension=self.embedding_dimension) 
            for _ in range(self.num_heads)
            ]


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # input is (B, context_length, embed_dim)
        # and output is (B, context_length, embed_dim)
        return torch.concat([attn(x, self.mask) for attn in self.maksed_attns], dim=-1)
        

In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, context_length: int, head_dim:int) -> None:
        super().__init__()
        self.context_length = context_length
        self.head_dim = head_dim

        pe = torch.zeros(self.context_length, self.head_dim)
        positions = torch.arange(self.context_length).unsqueeze(1)
        second_term = torch.exp((torch.arange(self.head_dim, 2)*(-math.log(1000)))/self.head_dim)
        pe[:, ::2]  = torch.sin(positions * second_term)
        pe[:, 1::2] = torch.cos(positions * second_term)        
        self.register_buffer("pe", pe)


    def forward(self, x:torch.Tensor) -> torch.Tensor:
        # shape of x is (B, Context Length, D_model)
        B, T, _ = x.shape
        assert T == self.context_length, "Context length should be same (Error in Position Encoding)"
        return x + self.pe[:self.context_length, :].unsqueeze(0) # type: ignore

        

In [ ]:
import math
class OnlineSoftmax(nn.Module):
    def __init__(self, tile_size: int) -> None:
        super().__init__()
        # Current Assumption and then I will make it parameterise
        self.tile_size = tile_size

    @torch.no_grad
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # shape of x is B, L, Head Dim
        assert x.ndim == 3, "online softmax expects 3 dimension input"
        context_length = x.shape[1]
        head_dim = x.shape[-1]
        assert head_dim % self.tile_size == 0, "context length should be divisible by tile size"

        running_max = torch.full((context_length, 1), -math.inf)
        denom = torch.full((context_length, 1), 0)
        for i in range(0, head_dim, self.tile_size):
            tile = x[..., i:i+self.tile_size]
            tile_max = torch.amax(tile, dim=-1, keepdim=True)
            new_max = torch.maximum(running_max, tile_max)
            alpha = torch.exp(running_max - new_max)
            denom = alpha * denom + torch.sum(torch.exp(tile-new_max), dim=-1, keepdim=True)
            running_max = new_max

        result = torch.exp(x-running_max) / denom
        return result


OnlineSoftmax(tile_size=2)(x=torch.randn((1, 8,24)))

tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]])


tensor([[[0.0154, 0.0521, 0.0141, 0.0329, 0.0261, 0.0101, 0.0266, 0.1087,
          0.0760, 0.0320, 0.0133, 0.0077, 0.0081, 0.0107, 0.0084, 0.0310,
          0.0206, 0.0044, 0.2705, 0.0151, 0.0377, 0.0616, 0.0590, 0.0577],
         [0.0454, 0.0393, 0.0199, 0.0115, 0.0227, 0.0129, 0.1198, 0.0032,
          0.0249, 0.0084, 0.2059, 0.0189, 0.0149, 0.0114, 0.0637, 0.0264,
          0.0171, 0.0655, 0.1589, 0.0096, 0.0320, 0.0074, 0.0200, 0.0406],
         [0.0196, 0.0202, 0.0394, 0.0553, 0.0482, 0.0281, 0.0762, 0.0382,
          0.0265, 0.0085, 0.0559, 0.0210, 0.0177, 0.0143, 0.0365, 0.0145,
          0.0383, 0.0212, 0.0284, 0.0080, 0.1322, 0.0231, 0.0337, 0.1949],
         [0.0998, 0.0130, 0.0784, 0.0438, 0.0109, 0.1107, 0.0193, 0.0235,
          0.0889, 0.0431, 0.0268, 0.0104, 0.0145, 0.0457, 0.0208, 0.0562,
          0.0096, 0.0086, 0.0324, 0.0708, 0.0326, 0.0310, 0.0758, 0.0333],
         [0.0180, 0.0109, 0.0043, 0.0389, 0.0296, 0.0324, 0.0044, 0.0061,
          0.0174, 0.4387, 0.0071, 

In [100]:

class FlashAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        # Inputs
        self.context_length = 8
        self.embedding_dim = 128
        self.head_dim = 32
        self.tile_size = 2
        self.x = torch.randn((self.context_length, self.embedding_dim)) # considering 2 dimensions only context_length, embedding_dim

        # Layers
        self.W_q = nn.Linear(self.embedding_dim, self.head_dim, bias=False)
        self.W_k = nn.Linear(self.embedding_dim, self.head_dim, bias=False)
        self.W_v = nn.Linear(self.embedding_dim, self.head_dim, bias=False)

    def forward(self) -> torch.Tensor:
        # context length is L
        # embedding dim is E
        # head_dim is D
        # tile size is T

        x = self.x.clone() # (L, E)
        q = self.W_q(x) # (L, D)
        k = self.W_k(x) # (L, D)
        v = self.W_v(x) # (L, D)

        output_buffer = torch.zeros((self.context_length, self.head_dim))
        # create a range of context length with step size is tile size
        for i in range(0, self.context_length, self.tile_size):
            q_tile = q[...,i:i+self.tile_size,:] # (T, D)

            running_max = torch.full((self.tile_size, 1), -math.inf)
            running_denom = torch.zeros((self.tile_size, 1))
            running_nume = torch.zeros((self.tile_size, self.head_dim))

            # one query tile will multiply with each key tile
            # one query tile multiple with key tiles in tile by tile. at the end one query tile will multiply each key tile but in the inner for loop we are doing step by step
            # not at once
            # at then end we need attention scores of each position with respect every other position
            # in this code we are doing tile by tile
            for j in range(0, self.context_length, self.tile_size):
                k_tile = k[...,j:j+self.tile_size,:] # (T, D)
                v_tile = v[...,j:j+self.tile_size,:]
                attn_scores = (q_tile @ k_tile.transpose(-2, -1)/math.sqrt(self.head_dim)) #  (T, T)

                new_max = torch.maximum(running_max, torch.amax(attn_scores, dim=-1, keepdim=True))
                alpha = torch.exp(running_max - new_max)

                running_denom = alpha * running_denom + torch.sum(torch.exp(attn_scores-new_max), dim=-1, keepdim=True)
                

                running_nume = alpha * running_nume + (torch.exp(attn_scores-new_max)@v_tile)
                running_max = new_max
            output_buffer[...,i:i+self.tile_size,:] = running_nume / running_denom

        return output_buffer
            

xx = FlashAttention()()
# print(xx)

In [ ]:
class ExpertFFN(nn.Module):
    def __init__(self, embedding_dimension:int) -> None:
        super().__init__()
        self.embedding_dimension = embedding_dimension
        self.linear1 = nn.Linear(self.embedding_dimension, 4*self.embedding_dimension)
        self.linear2 = nn.Linear(4*self.embedding_dimension, self.embedding_dimension)

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        return F.relu(x)


class TransformerBlock(nn.Module):
    def __init__(self, embedding_dimension:int, context_length: int, num_experts: int) -> None:
        super().__init__()
        self.num_experts = num_experts
        self.embedding_dimension = embedding_dimension
        self.context_length = context_length        
        self.heads = MultiHeadCasualAttention(num_heads=NUM_HEADS, embedding_dimension=self.embedding_dimension, context_length=self.context_length)
        self.router_layer = nn.Linear(embedding_dimension, NUM_EXPERTS, bias=False)
        self.experts = [ExpertFFN(embedding_dimension=self.embedding_dimension) for _ in range(self.num_experts)]

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        input_tensor = x
        assert x.shape == (BATCH_SIZE, CONTEXT_LENGTH, EMBEDDING_DIMENSION)
        x = self.heads(x)
        expert_selection = torch.argmax(F.softmax(self.router_layer(x), dim=-1)) # it gives us expert to handle each token (B, Context Len)
        x_flat = x.reshape(BATCH_SIZE * CONTEXT_LENGTH, EMBEDDING_DIMENSION)
        expert_selection_idx_flat = expert_selection.reshape(BATCH_SIZE * CONTEXT_LENGTH)

        out = torch.zeros_like(x_flat)
        for e, expert in enumerate(self.experts):
            mask = (expert_selection_idx_flat == e) # it is gathering all expert 0 , expert 1 and expert 2 and all other expert
            if mask.any():
                out[mask] = expert(x_flat[mask])
        out = out.reshape(BATCH_SIZE, CONTEXT_LENGTH, EMBEDDING_DIMENSION)


        return x

        

In [ ]:


class AbbyGPT(nn.Module):
    def __init__(self, vocab_size:int, embedding_dimension:int, context_length: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dimension = embedding_dimension
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dimension)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x is (batch_size, context_length)
        input_tensor = x
        x = self.embedding_layer(x) # (B, context_length, embed_dim)
        
        return x
